#1

In [2]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = PROJECT_ROOT / "data" / "ml-latest-small"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RATINGS_PATH = DATASET_DIR / "ratings.csv"
MOVIES_PATH = DATASET_DIR / "movies.csv"

print("Project Root:", PROJECT_ROOT)
print("Dataset Directory:", DATASET_DIR)
print("Processed Directory:", PROCESSED_DIR)

assert RATINGS_PATH.exists()
assert MOVIES_PATH.exists()


Project Root: c:\Users\yatha\Desktop\personalized-recommendation-engine
Dataset Directory: c:\Users\yatha\Desktop\personalized-recommendation-engine\data\ml-latest-small
Processed Directory: c:\Users\yatha\Desktop\personalized-recommendation-engine\data\processed


Load the raw datasets

In [3]:

ratings = pd.read_csv(RATINGS_PATH)
movies = pd.read_csv(MOVIES_PATH)

print("Ratings Shape:", ratings.shape)
print("Movies Shape:", movies.shape)

display(ratings.head())


Ratings Shape: (100836, 4)
Movies Shape: (9742, 3)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


Convert timestamps and validate data

In [4]:

ratings["timestamp"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

ratings = ratings.sort_values(
    ["userId", "timestamp"],
    kind="mergesort"
).reset_index(drop=True)

print("Timestamp Data Type:", ratings["timestamp"].dtype)
print("Earliest Interaction:", ratings["timestamp"].min())
print("Latest Interaction:", ratings["timestamp"].max())

assert ratings["timestamp"].notna().all()
assert ratings["rating"].between(0.5, 5.0).all()
assert ratings.duplicated(subset=["userId", "movieId"]).sum() == 0

print("Data validation passed.")


Timestamp Data Type: datetime64[us, UTC]
Earliest Interaction: 1996-03-29 18:36:55+00:00
Latest Interaction: 2018-09-24 14:27:30+00:00
Data validation passed.


#2 Chronological train, validation, and test split

Split ratings per user

In [5]:

def chronological_split(user_ratings):
    user_ratings = user_ratings.sort_values(
        "timestamp",
        kind="mergesort"
    )

    n = len(user_ratings)

    n_train = int(n * 0.80)
    n_val = int(n * 0.10)

    train = user_ratings.iloc[:n_train]
    validation = user_ratings.iloc[n_train:n_train + n_val]
    test = user_ratings.iloc[n_train + n_val:]

    return train, validation, test


train_parts = []
validation_parts = []
test_parts = []

for user_id, user_data in ratings.groupby("userId", sort=True):

    train_user, val_user, test_user = chronological_split(user_data)

    train_parts.append(train_user)
    validation_parts.append(val_user)
    test_parts.append(test_user)

train_df = pd.concat(train_parts).reset_index(drop=True)

val_df = pd.concat(validation_parts).reset_index(drop=True)

test_df = pd.concat(test_parts).reset_index(drop=True)

print("Training Set:", train_df.shape)
print("Validation Set:", val_df.shape)
print("Test Set:", test_df.shape)

display(train_df.head())


Training Set: (80419, 4)
Validation Set: (9818, 4)
Test Set: (10599, 4)


,userId,movieId,rating,timestamp
0,1,804,4.0,2000-07-30 18:08:19+00:00
1,1,1210,5.0,2000-07-30 18:08:19+00:00
2,1,2018,5.0,2000-07-30 18:08:43+00:00
3,1,2628,4.0,2000-07-30 18:08:43+00:00
4,1,2826,4.0,2000-07-30 18:08:43+00:00


Verify the split

In [6]:

print("Total Ratings:", len(ratings))

print(
    "Total After Splitting:",
    len(train_df) + len(val_df) + len(test_df)
)

print("\nUnique Users:")
print("Train:", train_df["userId"].nunique())
print("Validation:", val_df["userId"].nunique())
print("Test:", test_df["userId"].nunique())

assert len(train_df) + len(val_df) + len(test_df) == len(ratings)

assert train_df["userId"].nunique() == ratings["userId"].nunique()
assert val_df["userId"].nunique() == ratings["userId"].nunique()
assert test_df["userId"].nunique() == ratings["userId"].nunique()

print("\nSplit verification passed.")


Total Ratings: 100836
Total After Splitting: 100836

Unique Users:
Train: 610
Validation: 610
Test: 610

Split verification passed.


Check chronological leakage

In [7]:

train_max = train_df.groupby("userId")["timestamp"].max()

val_min = val_df.groupby("userId")["timestamp"].min()

val_max = val_df.groupby("userId")["timestamp"].max()

test_min = test_df.groupby("userId")["timestamp"].min()

assert (train_max <= val_min).all()
assert (val_max <= test_min).all()

print("Chronological leakage checks passed.")


Chronological leakage checks passed.


#3 Create user and movie ID mappings

Build mappings

In [8]:

user_ids = sorted(ratings["userId"].unique())

movie_ids = sorted(movies["movieId"].unique())

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

user_mapping = pd.DataFrame({
    "userId": user_ids,
    "user_idx": range(len(user_ids))
})

movie_mapping = pd.DataFrame({
    "movieId": movie_ids,
    "movie_idx": range(len(movie_ids))
})

print("Mapped Users:", len(user_mapping))
print("Mapped Movies:", len(movie_mapping))

display(user_mapping.head())
display(movie_mapping.head())


Mapped Users: 610
Mapped Movies: 9742


,userId,user_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


,movieId,movie_idx
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


Add internal indices to the datasets

In [9]:

def add_indices(df):
    df = df.copy()

    df["user_idx"] = df["userId"].map(user_to_idx)

    df["movie_idx"] = df["movieId"].map(movie_to_idx)

    assert df["user_idx"].notna().all()
    assert df["movie_idx"].notna().all()

    df["user_idx"] = df["user_idx"].astype("int32")

    df["movie_idx"] = df["movie_idx"].astype("int32")

    return df


train_df = add_indices(train_df)

val_df = add_indices(val_df)

test_df = add_indices(test_df)

display(train_df.head())

print("User index range:", train_df["user_idx"].min(), "-", train_df["user_idx"].max())

print("Movie index range:", train_df["movie_idx"].min(), "-", train_df["movie_idx"].max())


,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,804,4.0,2000-07-30 18:08:19+00:00,0,632
1,1,1210,5.0,2000-07-30 18:08:19+00:00,0,911
2,1,2018,5.0,2000-07-30 18:08:43+00:00,0,1493
3,1,2628,4.0,2000-07-30 18:08:43+00:00,0,1979
4,1,2826,4.0,2000-07-30 18:08:43+00:00,0,2126


User index range: 0 - 609
Movie index range: 0 - 9741


#4 Create positive interactions for retrieval

Generate retrieval training data

In [11]:

POSITIVE_RATING_THRESHOLD = 4.0

retrieval_train_df = train_df[
    train_df["rating"] >= POSITIVE_RATING_THRESHOLD
].copy()

retrieval_train_df["label"] = 1

print("Total Training Interactions:", len(train_df))

print("Positive Retrieval Interactions:", len(retrieval_train_df))

print("Positive Interaction Percentage:",round(len(retrieval_train_df) / len(train_df) * 100,2),"%")

display(retrieval_train_df.head())


Total Training Interactions: 80419
Positive Retrieval Interactions: 39071
Positive Interaction Percentage: 48.58 %


,userId,movieId,rating,timestamp,user_idx,movie_idx,label
0,1,804,4.0,2000-07-30 18:08:19+00:00,0,632,1
1,1,1210,5.0,2000-07-30 18:08:19+00:00,0,911,1
2,1,2018,5.0,2000-07-30 18:08:43+00:00,0,1493,1
3,1,2628,4.0,2000-07-30 18:08:43+00:00,0,1979,1
4,1,2826,4.0,2000-07-30 18:08:43+00:00,0,2126,1


Calculate training-only movie popularity

In [12]:

movie_popularity = (
    train_df.groupby("movie_idx")
    .agg(
        interaction_count=("rating", "count"),
        average_rating=("rating", "mean")
    )
    .reset_index()
)

movie_popularity = movie_popularity.sort_values(
    "interaction_count",
    ascending=False
)

display(movie_popularity.head(20))


,movie_idx,interaction_count,average_rating
302,314,303,4.156766
266,277,281,4.416370
247,257,271,4.175277
1864,1939,259,4.193050
489,510,251,4.163347
215,224,236,4.220339
96,97,224,4.017857
402,418,222,3.761261
0,0,201,3.915423
486,507,200,3.945000


#5 Save processed datasets

Save all datasets and mappings

In [13]:

train_df.to_csv(
    PROCESSED_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    PROCESSED_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR / "test.csv",
    index=False
)

retrieval_train_df.to_csv(
    PROCESSED_DIR / "retrieval_train.csv",
    index=False
)

user_mapping.to_csv(
    PROCESSED_DIR / "user_mapping.csv",
    index=False
)

movie_mapping.to_csv(
    PROCESSED_DIR / "movie_mapping.csv",
    index=False
)

movie_popularity.to_csv(
    PROCESSED_DIR / "movie_popularity.csv",
    index=False
)

print("All processed datasets saved successfully!")


All processed datasets saved successfully!


Final preprocessing verification

In [14]:

print("===== FINAL PREPROCESSING SUMMARY =====")

print("Training Interactions:", len(train_df))
print("Validation Interactions:", len(val_df))
print("Test Interactions:", len(test_df))

print("Total Users:", len(user_mapping))
print("Total Movies:", len(movie_mapping))

print("Positive Retrieval Interactions:", len(retrieval_train_df))

print("\nProcessed Files:")

for file in sorted(PROCESSED_DIR.glob("*.csv")):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

assert len(train_df) + len(val_df) + len(test_df) == len(ratings)

assert retrieval_train_df["rating"].min() >= POSITIVE_RATING_THRESHOLD

assert train_df["user_idx"].notna().all()
assert train_df["movie_idx"].notna().all()

print("\nNotebook 02 completed successfully!")


===== FINAL PREPROCESSING SUMMARY =====
Training Interactions: 80419
Validation Interactions: 9818
Test Interactions: 10599
Total Users: 610
Total Movies: 9742
Positive Retrieval Interactions: 39071

Processed Files:
movie_mapping.csv - 108.6 KB
movie_popularity.csv - 132.62 KB
retrieval_train.csv - 1915.61 KB
test.csv - 504.12 KB
train.csv - 3792.37 KB
user_mapping.csv - 5.17 KB
validation.csv - 465.93 KB

Notebook 02 completed successfully!
